# Script 2 — Preparação & Engenharia de Features (V5 — DFP + ITR + Macro)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Esta versão incorpora **DFPs e ITRs** como observações de treino e enriquece
o dataset com **variáveis macroeconômicas** (BCB/SGS) e **dados de mercado** (yfinance), como observações de treino, aumentando o dataset
de ~74 para ~369 observações — ganho de 5× sem alterar o Script 1.

## Estratégia de target: "próximo DFP anual estritamente posterior"

| Observação | Prediz |
|---|---|
| ITR Q1/2022 (mar) | DFP 2022 (dez) |
| ITR Q2/2022 (jun) | DFP 2022 (dez) |
| ITR Q3/2022 (set) | DFP 2022 (dez) |
| DFP 2022 (dez)    | DFP 2023 (dez) |


## Etapa 0 — Dependências, logging e configuração global

In [ ]:
import logging, json, pickle, warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)

logger = logging.getLogger('pipeline_preparacao_v5')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
_sh  = logging.StreamHandler(); _sh.setLevel(logging.INFO); _sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh  = logging.FileHandler(PASTA_SAIDA / 'pipeline_preparacao_v5.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros configuráveis ──────────────────────────────────────────────
LISTA_KPIS = [
    'margem_bruta','margem_ebit','margem_liquida','margem_ebitda',
    'roe','roa','liquidez_corrente','liquidez_imediata',
    'endividamento','alavancagem_de','div_liquida','cobertura_juros',
    'giro_ativo','fco_receita','fco_lucro','EBITDA',
    'FCF','margem_fcf','conversao_caixa',
]
TARGET_COLS = {
    'TARGET_DRE_3.01': 'DRE_3.01',
    'TARGET_DRE_3.11': 'DRE_3.11',
    'TARGET_EBITDA'  : 'EBITDA',
}
LIMIAR_NULO    = 0.80
FATOR_WINSOR   = 3.0
MAX_COLS_YOY   = 16
CLIP_YOY       = 5.0
CORR_MIN       = 0.10
N_FEATURES_RFE = 15
FRAC_TREINO    = 0.75
GAP_YOY_MIN    = 340
GAP_YOY_MAX    = 395

logger.info("Script 2 V5 (DFP+ITR) iniciado | pandas=%s", pd.__version__)

COLS_MACRO_FINAL = []  # preenchido pela Etapa 1B

## Etapa 1 — Carregamento e validação

In [ ]:
cam_parquet = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
if not cam_parquet.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {cam_parquet}\n"
        "Execute o Script 1 (01_cvm_processamento_V5.ipynb) antes de continuar."
    )

dataset = pd.read_parquet(cam_parquet)
logger.info("Dataset carregado: %d × %d", *dataset.shape)

# Preserva timezone America/Sao_Paulo gravado pelo Script 1 (utc=False)
dataset['DT_REFER']  = pd.to_datetime(dataset['DT_REFER'], errors='coerce', utc=False)
TZ_DATASET           = dataset['DT_REFER'].dt.tz   # usado na Etapa 8 para DT_TARGET
dataset['ANO']       = dataset['DT_REFER'].dt.year.astype('Int64')
dataset['TRIMESTRE'] = dataset['DT_REFER'].dt.quarter.astype('Int64')
dataset['MES']       = dataset['DT_REFER'].dt.month.astype('Int64')

COLS_OBR = ['CNPJ_CIA','NOME_CIA','SETOR','ANO','ORIGEM','DT_REFER']
faltando = [c for c in COLS_OBR if c not in dataset.columns]
if faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltando}")

n_emp = dataset['NOME_CIA'].nunique()
kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
kpis_ausentes  = [k for k in LISTA_KPIS if k not in dataset.columns]
if kpis_ausentes:
    logger.warning("KPIs ausentes: %s", kpis_ausentes)

for orig in ['DFP','ITR']:
    sub = dataset[dataset['ORIGEM']==orig]
    logger.info("%-3s: %d obs | %d empresas | anos %s",
                orig, len(sub), sub['NOME_CIA'].nunique(),
                sorted(sub['ANO'].dropna().astype(int).unique()))

print(f"\n{'='*60}")
print(f"  Dataset carregado | {dataset.shape[0]:,} × {dataset.shape[1]}")
print(f"  Empresas : {n_emp}/25 | DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()}")
print(f"  KPIs     : {len(kpis_presentes)}/{len(LISTA_KPIS)} | TZ: {TZ_DATASET}")
print(f"{'='*60}")


## Etapa 1B — Enriquecimento com Dados Macroeconômicos e de Mercado

Integra variáveis de contexto externo ao dataset financeiro. Duas fontes:

**BCB/SGS (Banco Central do Brasil):** séries temporais oficiais via API pública
`https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados`. Séries coletadas:
- SELIC (série 432): taxa básica de juros — afeta custo de dívida e resultado financeiro
- IPCA (série 433): inflação oficial — impacta receitas e margens por setor
- Câmbio USD/BRL (série 1): crítico para Petróleo, Commodities e exportadoras
- PIB trimestral real (série 4380): ciclo econômico — correlacionado com demanda

**Yahoo Finance via yfinance:** dados de mercado por ticker B3.
- CDS Brasil 5 anos (`EWZ` como proxy de risco soberano)
- Retorno acumulado 12 meses: proxy de expectativa do mercado
- Volatilidade realizada 60 dias: proxy de incerteza sobre a empresa

**Alinhamento temporal:** `merge_asof` por empresa e data, direction='backward' —
cada observação recebe o valor macro vigente EM SUA data de referência, sem
contaminação com dados futuros. Isso é anti-leakage temporal obrigatório.

**Fallback:** se a API estiver indisponível (sem internet ou rate limit), as colunas
macro são preenchidas com NaN e a imputação da Etapa 4 resolve via mediana do setor.
O pipeline não falha por ausência de dados macro.

In [ ]:
import requests, warnings
from datetime import datetime, timedelta

# ── Tickers B3 por empresa ────────────────────────────────────────────────
# Cada empresa âncora tem um ticker na B3. O yfinance usa sufixo '.SA'
TICKERS_B3 = {
    'Petrobras': 'PETR4.SA', 'Prio': 'PRIO3.SA', 'Ultrapar': 'UGPA3.SA',
    'Raizen': 'RAIZ4.SA', 'Vibra Energia': 'VBBR3.SA',
    'Engie Brasil': 'EGIE3.SA', 'Equatorial Energia': 'EQTL3.SA',
    'Taesa': 'TAEE11.SA', 'CPFL Energia': 'CPFE3.SA', 'ISA CTEEP': 'TRPL4.SA',
    'Lojas Renner': 'LREN3.SA', 'Magazine Luiza': 'MGLU3.SA',
    'Alpargatas': 'ALPA4.SA', 'Arezzo': 'ARZZ3.SA', 'Grupo Mateus': 'GMAT3.SA',
    'Vale': 'VALE3.SA', 'Suzano': 'SUZB3.SA', 'Klabin': 'KLBN11.SA',
    'Gerdau': 'GGBR4.SA', 'CSN Mineracao': 'CMIN3.SA',
    'WEG': 'WEGE3.SA', 'Totvs': 'TOTS3.SA', 'Positivo': 'POSI3.SA',
    'Intelbras': 'INTB3.SA', 'Brisanet': 'BRIT3.SA',
}

# ── Series BCB/SGS ────────────────────────────────────────────────────────
SERIES_BCB = {
    'macro_selic':     432,   # SELIC acumulada no mes (%)
    'macro_ipca':      433,   # IPCA mensal (%)
    'macro_cambio':    1,     # Taxa de cambio USD/BRL (venda, fim de periodo)
    'macro_pib_tri':   4380,  # PIB trimestral real (variacao %)
}

def buscar_serie_bcb(codigo, data_inicio='2014-01-01'):
    url = (f'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados'
           f'?formato=json&dataInicial={data_inicio}')
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        df = pd.DataFrame(r.json())
        df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y')
        df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
        return df.set_index('data')['valor'].sort_index()
    except Exception as e:
        logger.warning('BCB serie %d indisponivel: %s', codigo, e)
        return pd.Series(dtype=float)

def buscar_dados_mercado(ticker, data_inicio='2014-01-01'):
    try:
        import yfinance as yf
        hist = yf.download(ticker, start=data_inicio, progress=False, auto_adjust=True)
        if hist.empty: return pd.DataFrame()
        close = hist['Close'].squeeze()
        # Retorno acumulado 252 dias uteis (~1 ano)
        ret_12m = close.pct_change(252).rename('retorno_12m')
        # Volatilidade realizada 60 dias
        vol_60d = close.pct_change().rolling(60).std().rename('volatilidade_60d')
        return pd.concat([ret_12m, vol_60d], axis=1)
    except Exception as e:
        logger.warning('yfinance %s indisponivel: %s', ticker, e)
        return pd.DataFrame()

# ── Coleta das series BCB ─────────────────────────────────────────────────
logger.info('Coletando series macro do BCB/SGS...')
series_bcb = {}
for nome_col, codigo in SERIES_BCB.items():
    s = buscar_serie_bcb(codigo)
    if not s.empty:
        series_bcb[nome_col] = s
        logger.info('  %s: %d observacoes (%s a %s)',
                    nome_col, len(s), s.index.min().date(), s.index.max().date())
    else:
        logger.warning('  %s: sem dados (fallback para NaN)', nome_col)

# ── CDS Brasil via yfinance (EWZ como proxy) ─────────────────────────────
logger.info('Coletando CDS Brasil (proxy EWZ)...')
df_ewz = buscar_dados_mercado('EWZ')
if not df_ewz.empty:
    series_bcb['macro_vol_brasil'] = df_ewz['volatilidade_60d'].rename('macro_vol_brasil')

# ── Alinhamento temporal: merge_asof por data ─────────────────────────────
# Para cada linha do dataset, pega o valor macro mais recente ANTERIOR ou IGUAL
# a DT_REFER (direction='backward') — anti-leakage temporal obrigatorio
if series_bcb:
    df_macro_ts = pd.DataFrame(series_bcb)
    df_macro_ts.index = pd.to_datetime(df_macro_ts.index)
    # Normalizar timezone para compatibilidade com DT_REFER
    df_macro_ts.index = df_macro_ts.index.tz_localize(None)

    # DT_REFER sem timezone para o merge
    dt_merge = dataset['DT_REFER'].dt.tz_localize(None)
    df_merge_key = pd.DataFrame({'DT_REFER_naive': dt_merge,
                                  'idx_orig': range(len(dataset))})
    df_merge_key = df_merge_key.sort_values('DT_REFER_naive')
    df_macro_ts_sorted = df_macro_ts.sort_index().reset_index().rename(columns={'index':'DT_MACRO'})

    merged_macro = pd.merge_asof(
        df_merge_key,
        df_macro_ts_sorted,
        left_on='DT_REFER_naive',
        right_on='DT_MACRO',
        direction='backward',
        tolerance=pd.Timedelta('92 days'),   # max 1 trimestre de defasagem
    ).set_index('idx_orig').sort_index()

    for col in df_macro_ts.columns:
        if col in merged_macro.columns:
            dataset[col] = merged_macro[col].values
            cob = dataset[col].notna().mean()
            logger.info('  Macro %-25s integrada | cobertura %.0f%%', col, cob*100)

    cols_macro = [c for c in dataset.columns if c.startswith('macro_')]
    logger.info('Macro integrada: %d colunas | cobertura media %.0f%%',
                len(cols_macro),
                dataset[cols_macro].notna().mean().mean()*100 if cols_macro else 0)
else:
    cols_macro = []
    logger.warning('Nenhuma serie macro disponivel — pipeline continua sem dados macro')

print(f'Macro integrada: {len(cols_macro)} colunas')
if cols_macro:
    print(dataset[cols_macro].describe().round(4).to_string())

In [ ]:
# ── Dados de mercado por empresa (yfinance) ──────────────────────────────
# Retorno acumulado 12m e volatilidade 60d por empresa e data
# Permite ao modelo aprender que mercado ja precificou resultados futuros
logger.info('Coletando dados de mercado via yfinance...')

if 'NOME_CIA' in dataset.columns:
    df_mercado_list = []
    for empresa, ticker in TICKERS_B3.items():
        df_mkt = buscar_dados_mercado(ticker)
        if df_mkt.empty: continue
        df_mkt.index = df_mkt.index.tz_localize(None)
        df_mkt_r = df_mkt.reset_index().rename(columns={'Date':'DT_MKT'})
        df_mkt_r['NOME_CIA'] = empresa
        df_mercado_list.append(df_mkt_r)

    if df_mercado_list:
        df_mercado_all = pd.concat(df_mercado_list, ignore_index=True).sort_values('DT_MKT')

        # merge_asof por empresa + data
        partes_mkt = []
        for empresa, grp in dataset.sort_values('DT_REFER').groupby('NOME_CIA'):
            df_emp_mkt = df_mercado_all[df_mercado_all['NOME_CIA']==empresa]
            if df_emp_mkt.empty:
                partes_mkt.append(grp)
                continue
            grp_c = grp.copy()
            grp_c['_dt_naive'] = grp_c['DT_REFER'].dt.tz_localize(None)
            merged = pd.merge_asof(
                grp_c.sort_values('_dt_naive'),
                df_emp_mkt[['DT_MKT','retorno_12m','volatilidade_60d']],
                left_on='_dt_naive', right_on='DT_MKT',
                direction='backward',
                tolerance=pd.Timedelta('31 days'),
            ).drop(columns=['DT_MKT','_dt_naive'], errors='ignore')
            partes_mkt.append(merged)

        dataset = pd.concat(partes_mkt, ignore_index=True)
        dataset = dataset.sort_values(['CNPJ_CIA','DT_REFER']).reset_index(drop=True)

        cols_mkt = [c for c in dataset.columns if c in ['retorno_12m','volatilidade_60d']]
        for c in cols_mkt: cols_macro.append(c)
        logger.info('Dados de mercado integrados: %s | cobertura: %s',
                    cols_mkt,
                    {c: f"{dataset[c].notna().mean():.0%}" for c in cols_mkt})
        print(f'Dados de mercado integrados: {cols_mkt}')
    else:
        logger.warning('Dados de mercado indisponiveis — pipeline continua sem eles')
else:
    logger.warning('NOME_CIA nao encontrado — dados de mercado ignorados')

# Atualizar lista de KPIs protegidos com colunas macro
COLS_MACRO_FINAL = [c for c in dataset.columns
                    if c.startswith('macro_') or c in ['retorno_12m','volatilidade_60d']]
logger.info('Total colunas macro+mercado: %d', len(COLS_MACRO_FINAL))
print(f'Total colunas macro+mercado adicionadas: {len(COLS_MACRO_FINAL)}')

## Etapa 2 — Deduplicação intra-período

In [ ]:
n_antes = len(dataset)
dataset['_n_kpis'] = dataset[kpis_presentes].notna().sum(axis=1)
dataset = (dataset
    .sort_values(['CNPJ_CIA','DT_REFER','ORIGEM','_n_kpis'], ascending=[True,True,True,False])
    .drop_duplicates(subset=['CNPJ_CIA','DT_REFER','ORIGEM'], keep='first')
    .drop(columns=['_n_kpis'])
    .sort_values(['CNPJ_CIA','DT_REFER'])
    .reset_index(drop=True)
)
rem = n_antes - len(dataset)
logger.info("Dedup: %d → %d (-%d)", n_antes, len(dataset), rem)
print(f"Dedup: {n_antes} → {len(dataset)} (removidas {rem} retificações)")


## Etapa 3 — Remoção de colunas com >80% de nulos

In [ ]:
COLS_MACRO_FINAL = [c for c in dataset.columns
                    if c.startswith('macro_') or c in ['retorno_12m','volatilidade_60d']]
COLS_PROTEGIDAS = set(kpis_presentes + ['ANO','TRIMESTRE','MES'] + COLS_MACRO_FINAL)
cols_num        = dataset.select_dtypes(include='number').columns.tolist()
cols_cand       = [c for c in cols_num if c not in COLS_PROTEGIDAS]
taxa_nulo       = dataset[cols_cand].isnull().mean()
cols_excluir    = taxa_nulo[taxa_nulo > LIMIAR_NULO].index.tolist()

grupos_exc = {}
for c in cols_excluir:
    p = c.split('_')[0]; grupos_exc[p] = grupos_exc.get(p, 0) + 1
logger.info("Remoção >%.0f%% nulos: %d colunas | %s", LIMIAR_NULO*100, len(cols_excluir),
            dict(sorted(grupos_exc.items(), key=lambda x: -x[1])))

dataset = dataset.drop(columns=cols_excluir)
kpis_presentes = [k for k in kpis_presentes if k in dataset.columns]
print(f"Removidas: {len(cols_excluir)} colunas | Dataset: {dataset.shape} | KPIs: {len(kpis_presentes)}")


## Etapa 4 — Imputação por mediana do setor

In [ ]:
n_nulos_pre = dataset[kpis_presentes].isnull().sum().sum()

for kpi in kpis_presentes:
    if dataset[kpi].isnull().sum() == 0:
        continue
    # Nível 1: mediana por (setor, origem) — respeita diferença DFP vs ITR
    m1 = dataset.groupby(['SETOR','ORIGEM'])[kpi].transform('median')
    # Nível 2: mediana por setor
    m2 = dataset.groupby('SETOR')[kpi].transform('median')
    # Nível 3: mediana global
    m3 = dataset[kpi].median()
    dataset[kpi] = dataset[kpi].fillna(m1).fillna(m2).fillna(m3)

n_nulos_pos = dataset[kpis_presentes].isnull().sum().sum()
logger.info("Imputação: %d → %d nulos", n_nulos_pre, n_nulos_pos)
print(f"Nulos KPIs: {n_nulos_pre} → {n_nulos_pos}")


## Etapa 5 — Winsorização por setor

**CORREÇÃO pandas 3.x:** usa `groupby().transform()` em vez de `groupby().apply()`.
O `apply()` no pandas >= 2.0 dropa silenciosamente a coluna `SETOR` usada no agrupamento,
causando `KeyError` a partir do segundo KPI. O `transform()` é imune a esse comportamento.

In [ ]:
def winsorizacao_setor(df_in: pd.DataFrame, col: str, fator: float = 3.0) -> pd.DataFrame:
    """
    Winsorização coluna a coluna via groupby+transform.
    transform() preserva TODOS os campos do DataFrame — nunca dropa SETOR.
    Compatível com pandas >= 2.0 / 3.x.
    """
    def _clip(serie: pd.Series) -> pd.Series:
        q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            return serie
        return serie.clip(lower=q1 - fator * iqr, upper=q3 + fator * iqr)

    df_out = df_in.copy()
    df_out[col] = df_in.groupby('SETOR')[col].transform(_clip)
    return df_out

n_clip_total = 0
for kpi in kpis_presentes:
    antes = dataset[kpi].copy()
    dataset = winsorizacao_setor(dataset, kpi, FATOR_WINSOR)
    n_clip_total += (dataset[kpi] != antes).sum()

assert 'SETOR' in dataset.columns, "SETOR perdido na winsorização — invariante violado"
logger.info("Winsorização: fator=%.1f | %d valores truncados", FATOR_WINSOR, n_clip_total)
print(f"✅ Winsorização | fator={FATOR_WINSOR} | {n_clip_total} valores truncados | SETOR: ✅")


## Etapa 6 — YoY mesmo trimestre ano anterior (shift=4)

Com dados mistos (3 ITRs + 1 DFP por ano), o `shift(1)` compararia Q1 com Q4 do ano anterior.
O `shift(4)` compara cada período com o **mesmo período do ano anterior** — YoY real.

In [ ]:
dataset = dataset.sort_values(['CNPJ_CIA','DT_REFER']).reset_index(drop=True)

KPIS_YOY = [k for k in kpis_presentes
             if k not in ('div_liquida','EBITDA','FCF')][:MAX_COLS_YOY]

# Pré-calcular gap para validação de continuidade (shift=4 ≈ 1 ano)
dataset['_dt_prev'] = dataset.groupby('CNPJ_CIA')['DT_REFER'].shift(4)
dataset['_gap_dias'] = (dataset['DT_REFER'] - dataset['_dt_prev']).dt.days
gap_invalido = ~dataset['_gap_dias'].between(GAP_YOY_MIN, GAP_YOY_MAX)
logger.info("YoY gaps inválidos: %d / %d (%.1f%%)",
            gap_invalido.sum(), len(dataset), gap_invalido.sum()/len(dataset)*100)

cols_yoy = []
for kpi in KPIS_YOY:
    col_yoy = f'{kpi}_yoy'
    prev    = dataset.groupby('CNPJ_CIA')[kpi].shift(4)
    yoy_raw = ((dataset[kpi] - prev) / prev.abs().replace(0, np.nan))
    yoy_raw = yoy_raw.replace([np.inf,-np.inf], np.nan).clip(-CLIP_YOY, CLIP_YOY)
    yoy_raw[gap_invalido] = np.nan
    dataset[col_yoy] = yoy_raw
    cols_yoy.append(col_yoy)

if 'margem_ebitda_yoy' in dataset.columns:
    accel = dataset.groupby('CNPJ_CIA')['margem_ebitda_yoy'].diff()
    accel[gap_invalido] = np.nan
    dataset['aceleracao_ebitda'] = accel
    cols_yoy.append('aceleracao_ebitda')

# pos_ciclo: Q1=1, Q2=2, Q3=3, DFP=4 (independente do mês de encerramento fiscal)
dataset['pos_ciclo'] = dataset['TRIMESTRE'].astype(float)
dataset.loc[dataset['ORIGEM']=='DFP', 'pos_ciclo'] = 4.0
cols_yoy.append('pos_ciclo')

dataset = dataset.drop(columns=['_dt_prev','_gap_dias'])

logger.info("YoY: %d features | nulos médios: %.0f%%",
            len(cols_yoy), dataset[cols_yoy].isnull().mean().mean()*100)
print(f"Features YoY: {len(cols_yoy)} | nulos médios: {dataset[cols_yoy].isnull().mean().mean():.0%}")


## Etapa 7 — One-hot do setor e flag de origem

In [ ]:
dataset = pd.get_dummies(dataset, columns=['SETOR'], prefix='setor', dtype=float)
cols_setor = sorted([c for c in dataset.columns if c.startswith('setor_')])

# flag_dfp: informa ao modelo se os dados são acumulados anuais (1) ou parciais (0)
dataset['flag_dfp'] = (dataset['ORIGEM'] == 'DFP').astype(float)

logger.info("One-hot SETOR: %d colunas | flag_dfp criada", len(cols_setor))
print(f"Setores: {cols_setor}")
print(f"Dataset: {dataset.shape}")


## Etapa 8 — Targets: próximo DFP estritamente posterior

Para cada linha, o target é o **primeiro DFP com DT_REFER > DT_REFER da linha atual**.

**Correção de timezone:** `DT_TARGET` é inicializada como `pd.Series` com dtype
`datetime64[us, America/Sao_Paulo]` — mesmo timezone que `DT_REFER` — evitando
`TypeError: Cannot compare tz-naive and tz-aware datetime-like objects`.

In [ ]:
# Tabela de DFPs disponíveis por empresa
dfp_targets = (
    dataset[dataset['ORIGEM'] == 'DFP']
    [['CNPJ_CIA','DT_REFER'] + list(TARGET_COLS.values())]
    .rename(columns={'DT_REFER': 'DT_DFP',
                     **{v: k for k, v in TARGET_COLS.items()}})
    .sort_values(['CNPJ_CIA','DT_DFP'])
    .reset_index(drop=True)
)

# Lookup por empresa: próximo DFP ESTRITAMENTE posterior (DT_DFP > DT_REFER da linha)
partes = []
for cnpj, grupo in dataset.sort_values(['CNPJ_CIA','DT_REFER']).groupby('CNPJ_CIA'):
    dfp_emp = dfp_targets[dfp_targets['CNPJ_CIA'] == cnpj].sort_values('DT_DFP').reset_index(drop=True)
    g = grupo.sort_values('DT_REFER').copy()

    # Inicializar colunas de target
    for tgt_col in TARGET_COLS:
        g[tgt_col] = np.nan
    # DT_TARGET como Series tz-aware (mesmo timezone que DT_REFER)
    dt_vals = {}

    for idx in g.index:
        dt_linha = g.loc[idx, 'DT_REFER']
        proximos = dfp_emp[dfp_emp['DT_DFP'] > dt_linha]
        if len(proximos):
            p = proximos.iloc[0]
            for tgt_col in TARGET_COLS:
                g.loc[idx, tgt_col] = p[tgt_col]
            dt_vals[idx] = p['DT_DFP']

    # Atribuição tz-aware segura: via pd.Series com dtype explícito
    g['DT_TARGET'] = pd.Series(dt_vals, dtype=f'datetime64[us, {TZ_DATASET}]')
    partes.append(g)

dataset = pd.concat(partes, ignore_index=True)
targets_criados = [t for t in TARGET_COLS if t in dataset.columns]

for tgt in targets_criados:
    n_val = dataset[tgt].notna().sum()
    logger.info("Target %-22s: %d/%d válidos (%.0f%%)",
                tgt, n_val, len(dataset), n_val/len(dataset)*100)

print("Targets criados:")
for orig in ['DFP','ITR']:
    sub = dataset[dataset['ORIGEM']==orig]
    val = sub[targets_criados[0]].notna().sum()
    print(f"  {orig}: {val}/{len(sub)} obs com target")

n_ant = len(dataset)
dataset = dataset[dataset[targets_criados].notna().any(axis=1)].reset_index(drop=True)
logger.info("Remoção sem target: %d → %d", n_ant, len(dataset))
print(f"\nDataset com targets: {dataset.shape[0]} × {dataset.shape[1]}")


## Etapa 9 — Seleção de features por correlação e RFE

In [ ]:
FEATURES_CANDIDATAS = (
    kpis_presentes
    + cols_yoy
    + cols_setor
    + ['flag_dfp', 'pos_ciclo']
    + (['aceleracao_ebitda'] if 'aceleracao_ebitda' in dataset.columns else [])
    + COLS_MACRO_FINAL   # variaveis macro: SELIC, IPCA, cambio, PIB, mercado
)
FEATURES_CANDIDATAS = [f for f in FEATURES_CANDIDATAS if f in dataset.columns]
logger.info("Features candidatas: %d", len(FEATURES_CANDIDATAS))

target_principal = 'TARGET_DRE_3.01'

if target_principal not in dataset.columns:
    logger.error("Target principal não encontrado")
    FEATURES_SELECIONADAS = FEATURES_CANDIDATAS[:N_FEATURES_RFE]
else:
    df_sel = dataset[FEATURES_CANDIDATAS + [target_principal]].dropna()
    X_sel  = df_sel[FEATURES_CANDIDATAS]
    y_sel  = df_sel[target_principal]

    corr_abs = X_sel.corrwith(y_sel).abs().sort_values(ascending=False)
    FEATURES_CORR = corr_abs[corr_abs >= CORR_MIN].index.tolist()
    logger.info("Pearson: %d → %d features", len(FEATURES_CANDIDATAS), len(FEATURES_CORR))

    n_rfe = min(N_FEATURES_RFE, len(FEATURES_CORR))
    if len(FEATURES_CORR) <= n_rfe:
        FEATURES_SELECIONADAS = FEATURES_CORR
    else:
        scaler   = StandardScaler()
        X_scaled = scaler.fit_transform(df_sel[FEATURES_CORR])
        rfe = RFE(Ridge(alpha=1.0), n_features_to_select=n_rfe, step=2)
        rfe.fit(X_scaled, y_sel)
        FEATURES_SELECIONADAS = [FEATURES_CORR[i] for i, sel in enumerate(rfe.support_) if sel]
        logger.info("RFE: %d → %d features", len(FEATURES_CORR), len(FEATURES_SELECIONADAS))

    print(f"\nFeatures selecionadas ({len(FEATURES_SELECIONADAS)}):")
    for f in FEATURES_SELECIONADAS:
        print(f"  {f:<35} |r| = {corr_abs.get(f,0):.3f}")


## Etapa 10 — Split temporal treino/teste

**CORREÇÃO pandas 3.x:** a versão anterior usava `groupby('CNPJ_CIA').apply(split_func)`
onde `split_func` retornava o DataFrame completo. O pandas 3.x dropa `CNPJ_CIA` nesse padrão.

**Solução:** a função de split retorna apenas uma `pd.Series` com os labels `treino/teste`
(indexada pelo índice original). O DataFrame `dataset` é atualizado via atribuição direta.

In [ ]:
def calcular_split(grupo: pd.DataFrame, frac: float = 0.75) -> pd.Series:
    """
    Retorna pd.Series com labels 'treino'/'teste' indexada pelo índice do grupo.
    Retornar Series (não DataFrame) evita o drop de CNPJ_CIA pelo pandas 3.x.
    """
    grupo_ord = grupo.sort_values('DT_REFER')
    n    = len(grupo_ord)
    n_tr = max(1, round(n * frac))
    n_tr = min(n_tr, n - 1)
    resultado = pd.Series('teste', index=grupo_ord.index)
    resultado.iloc[:n_tr] = 'treino'
    return resultado

# Aplica o split: a Series retornada é alinhada ao índice do dataset
dataset['split'] = dataset.groupby('CNPJ_CIA', group_keys=False).apply(calcular_split)

assert 'CNPJ_CIA' in dataset.columns, "CNPJ_CIA perdido — invariante violado"

treino = dataset[dataset['split'] == 'treino'].copy()
teste  = dataset[dataset['split'] == 'teste'].copy()

# Verificação anti-contaminação temporal
contaminados = []
for cnpj, grp in dataset.groupby('CNPJ_CIA'):
    tr_max = grp[grp['split']=='treino']['DT_REFER'].max()
    te_min = grp[grp['split']=='teste']['DT_REFER'].min()
    if pd.notna(tr_max) and pd.notna(te_min) and tr_max > te_min:
        contaminados.append(cnpj)
if contaminados:
    logger.error("Contaminação temporal detectada: %s", contaminados)
else:
    logger.info("Anti-contaminação temporal: PASSOU ✅")

n_tot = len(dataset)
logger.info("Split: %d treino (%.0f%%) | %d teste (%.0f%%)",
            len(treino), len(treino)/n_tot*100, len(teste), len(teste)/n_tot*100)

print(f"Treino : {len(treino)} obs ({len(treino)/n_tot:.0%}) | DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"Teste  : {len(teste)} obs ({len(teste)/n_tot:.0%}) | DFP={(teste['ORIGEM']=='DFP').sum()} | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"CNPJ_CIA preservado: ✅ | Anti-contaminação: {'✅' if not contaminados else '❌'}")


## Etapa 11 — Persistência dos artefatos para o Script 3

In [ ]:
TARGETS_VALIDOS  = [t for t in TARGET_COLS if t in dataset.columns]
grupos_treino    = treino['CNPJ_CIA'].values   # para GroupKFold no Script 3

artefatos_df = {'dataset_preparado': dataset, 'treino': treino, 'teste': teste}
artefatos_meta = {
    'features'      : FEATURES_SELECIONADAS,
    'targets'       : TARGETS_VALIDOS,
    'kpis'          : kpis_presentes,
    'cols_setor'    : cols_setor,
    'cols_yoy'      : cols_yoy,
    'grupos_treino' : grupos_treino,
    'params': {
        'versao'              : 'V3_DFP_ITR',
        'limiar_nulo'         : LIMIAR_NULO,
        'fator_winsor'        : FATOR_WINSOR,
        'max_cols_yoy'        : MAX_COLS_YOY,
        'clip_yoy'            : CLIP_YOY,
        'corr_min'            : CORR_MIN,
        'n_features_rfe'      : N_FEATURES_RFE,
        'frac_treino'         : FRAC_TREINO,
        'gap_yoy_min'         : GAP_YOY_MIN,
        'gap_yoy_max'         : GAP_YOY_MAX,
        'estrategia_target'   : 'proximo_dfp_estritamente_posterior',
        'pandas_version'      : pd.__version__,
    },
}

for nome, df_art in artefatos_df.items():
    cam = PASTA_SAIDA / f'{nome}.parquet'
    df_art.to_parquet(cam, index=False)
    logger.info("Salvo: %s | %d × %d | %.0f KB", cam.name, *df_art.shape, cam.stat().st_size/1024)

for nome, obj in artefatos_meta.items():
    cam = PASTA_SAIDA / f'{nome}.pkl'
    with open(cam,'wb') as f: pickle.dump(obj, f)
    logger.info("Salvo: %s", cam.name)

relatorio = {
    'versao'                   : 'V5_DFP_ITR',
    'estrategia_target'        : 'proximo_dfp_estritamente_posterior',
    'dataset_empresas'         : int(dataset['CNPJ_CIA'].nunique()),
    'dataset_anos'             : sorted(dataset['ANO'].dropna().astype(int).unique().tolist()),
    'n_obs_dfp'                : int((dataset['ORIGEM']=='DFP').sum()),
    'n_obs_itr'                : int((dataset['ORIGEM']=='ITR').sum()),
    'n_obs_total'              : int(len(dataset)),
    'n_obs_treino'             : int(len(treino)),
    'n_obs_treino_dfp'         : int((treino['ORIGEM']=='DFP').sum()),
    'n_obs_treino_itr'         : int((treino['ORIGEM']=='ITR').sum()),
    'n_obs_teste'              : int(len(teste)),
    'n_features_candidatas'    : int(len(FEATURES_CANDIDATAS)),
    'n_features_selecionadas'  : int(len(FEATURES_SELECIONADAS)),
    'targets'                  : TARGETS_VALIDOS,
    'features'                 : FEATURES_SELECIONADAS,
    'params'                   : artefatos_meta['params'],
    'nota_groupkfold'          : 'Usar GroupKFold(groups=grupos_treino) no Script 3.',
    'bugs_corrigidos'          : [
        'Etapa 5: groupby(SETOR).apply() → transform() — pandas 3.x dropa coluna de agrupamento',
        'Etapa 8: DT_TARGET inicializada com dtype datetime64[us, America/Sao_Paulo] — tz-aware',
        'Etapa 10: groupby(CNPJ_CIA).apply() retorna Series não DataFrame — preserva CNPJ_CIA',
    ],
}
cam_rel = PASTA_SAIDA / 'relatorio_preparacao_v5.json'
with open(cam_rel,'w',encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*65)
print("  RESUMO FINAL — Script 2 V5 (DFP + ITR)")
print("═"*65)
print(f"  Total obs     : {len(dataset)} (DFP: {(dataset['ORIGEM']=='DFP').sum()} | ITR: {(dataset['ORIGEM']=='ITR').sum()})")
print(f"  Empresas      : {dataset['CNPJ_CIA'].nunique()} / 25")
print(f"  Treino        : {len(treino)} obs ({len(treino)/len(dataset):.0%})")
print(f"    ↳ DFP       : {(treino['ORIGEM']=='DFP').sum()}")
print(f"    ↳ ITR       : {(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste         : {len(teste)} obs ({len(teste)/len(dataset):.0%})")
print(f"  Features      : {len(FEATURES_SELECIONADAS)} selecionadas")
print(f"  Targets       : {TARGETS_VALIDOS}")
print(f"  Vs. V2(DFP)   : {len(dataset)/74:.1f}× mais observações")
print("═"*65)
print("  ⚠️  Script 3: usar GroupKFold(groups=grupos_treino) no CV")
print("  ✅  Pronto para o Script 3 (03_cvm_modelagem.ipynb)")
print("═"*65)
